In [ ]:
!pip install transformers sentencepiece pandas google-genai


In [ ]:
from transformers import AutoTokenizer
from google.colab import userdata
from google import genai
import pandas as pd

texts = {
    "English": "Generative AI models predict the next token in a sentence.",
    "Telugu": "జనరేటివ్ ఏఐ నమూనాలు వాక్యంలో తదుపరి టోకెన్‌ను అంచనా వేస్తాయి.",
    "Hindi": "जनरेटिव एआई मॉडल वाक्य में अगले टोकन का अनुमान लगाते हैं।",
}

tokenizers = {
    "BPE (gpt2)": "gpt2",
    "WordPiece (bert-base-multilingual-cased)": "bert-base-multilingual-cased",
    "SentencePiece (xlm-roberta-base)": "xlm-roberta-base",
}

rows = []

for name, model_id in tokenizers.items():
    tok = AutoTokenizer.from_pretrained(model_id)

    for lang, s in texts.items():
        ids = tok(s, add_special_tokens=False).input_ids

        rows.append({
            "tokenizer": name,
            "language": lang,
            "words": len(s.split()),
            "tokens": len(ids),
            "tokens_per_word": round(len(ids) / len(s.split()), 2)
        })

    print(name, "| vocab size:", tok.vocab_size)
    print("English pieces:", tok.tokenize(texts["English"])[:10])
    print("Telugu pieces:", tok.tokenize(texts["Telugu"])[:10])
    print("Hindi Token IDs:", tok(
        texts["Hindi"], add_special_tokens=False
    ).input_ids[:10])

# Gemini comparison
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

for lang, s in texts.items():
    n = client.models.count_tokens(
        model="gemini-3.5-flash-lite",
        contents=s
    ).total_tokens

    rows.append({
        "tokenizer": "Gemini",
        "language": lang,
        "words": len(s.split()),
        "tokens": n,
        "tokens_per_word": round(n / len(s.split()), 2)
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Token ratio
ratio = df.pivot(
    index="tokenizer",
    columns="language",
    values="tokens"
)

ratio["Telugu / English"] = (
    ratio["Telugu"] / ratio["English"]
).round(2)

ratio["Hindi / English"] = (
    ratio["Hindi"] / ratio["English"]
).round(2)

print("\nToken-ratio table:")
print(ratio.to_string())

BPE (gpt2) | vocab size: 50257
English pieces: ['Gener', 'ative', 'ĠAI', 'Ġmodels', 'Ġpredict', 'Ġthe', 'Ġnext', 'Ġtoken', 'Ġin', 'Ġa']
Telugu pieces: ['à', '°', 'ľ', 'à', '°', '¨', 'à', '°', '°', 'à']
Hindi Token IDs: [11976, 250, 11976, 101, 11976, 108, 24231, 229, 11976, 253]
WordPiece (bert-base-multilingual-cased) | vocab size: 119547
English pieces: ['Genera', '##tive', 'AI', 'models', 'pred', '##ict', 'the', 'next', 'tok', '##en']
Telugu pieces: ['జ', '##న', '##ర', '##ే', '##టి', '##వ్', 'ఏ', '##ఐ', 'న', '##మ']
Hindi Token IDs: [872, 11453, 28546, 14835, 66306, 860, 111193, 15801, 889, 111211]
SentencePiece (xlm-roberta-base) | vocab size: 250002
English pieces: ['▁Genera', 'tive', '▁AI', '▁models', '▁predict', '▁the', '▁next', '▁to', 'ken', '▁in']
Telugu pieces: ['▁జన', 'రే', 'టివ్', '▁ఏ', 'ఐ', '▁న', 'మూ', 'నాలు', '▁వా', 'క్']
Hindi Token IDs: [15103, 8503, 207628, 9489, 31046, 187543, 147131, 421, 127831, 197670]
                               tokenizer language  words  tokens